# Problem Set 4. Text III: attention & transformers

**Course:** Analysis of Unstructured Data  
**Instructor:** Janusz Szwabiński  
**Goal:** Understand and reproduce the attention mechanism, then use a pretrained transformer.  

---


In [ ]:
# Setup environment & imports
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


### Problem 1 [Theory] (Scaled dot-product attention)

**Task:**
Derive scaled dot-product attention $\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$. Give the variance argument for the $1/\sqrt{d_k}$ scaling (what happens to the softmax as $d_k$ grows without it?).

---

### Assumptions & Notation

1. **Input Matrices:**
   - Query matrix: $Q \in \mathbb{R}^{n \times d_k}$ with row vectors $q_i \in \mathbb{R}^{d_k}$ for $i \in \{1, \dots, n\}$.
   - Key matrix: $K \in \mathbb{R}^{m \times d_k}$ with row vectors $k_j \in \mathbb{R}^{d_k}$ for $j \in \{1, \dots, m\}$.
   - Value matrix: $V \in \mathbb{R}^{m \times d_v}$ with row vectors $v_j \in \mathbb{R}^{d_v}$ for $j \in \{1, \dots, m\}$.
   Here $n$ denotes query sequence length, $m$ denotes key/value sequence length ($n = m$ in self-attention), $d_k$ denotes the projection dimension of queries and keys, and $d_v$ denotes the value dimension.

2. **Probabilistic Modeling of Representation Vectors:**
   Assume the individual feature components $q_{i,l}$ and $k_{j,l}$ ($l \in \{1, \dots, d_k\}$) are independent, identically distributed (i.i.d.) random variables with zero mean and unit variance:
   $$\mathbb{E}[q_{i,l}] = 0, \quad \mathbb{E}[k_{j,l}] = 0$$
   $$\mathrm{Var}(q_{i,l}) = 1, \quad \mathrm{Var}(k_{j,l}) = 1$$
   Furthermore, queries and keys are mutually independent: $\mathbb{E}[q_{i,l} k_{j, l'}] = 0$ for all $l, l'$.

3. **Softmax Operator:**
   For any logit vector $z = (z_1, \dots, z_m)^\top \in \mathbb{R}^{m}$, the row-wise softmax operator $\mathrm{softmax}: \mathbb{R}^m \to \Delta^{m-1}$ maps $z$ onto the probability simplex:
   $$\mathrm{softmax}(z)_j = \frac{\exp(z_j)}{\sum_{l=1}^m \exp(z_l)}, \quad j \in \{1, \dots, m\}$$

---

### Formal Proposition

1. **Scaled Dot-Product Attention:**
   The output representation for query sequence $Q$ interacting with key-value pairs $(K, V)$ is given by:
   $$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$
   where the $i$-th row is the convex combination:
   $$\mathrm{Attn}(Q, K, V)_i = \sum_{j=1}^m A_{ij} v_j, \quad A_{ij} = \frac{\exp\left(\frac{q_i k_j^\top}{\sqrt{d_k}}\right)}{\sum_{l=1}^m \exp\left(\frac{q_i k_l^\top}{\sqrt{d_k}}\right)}$$

2. **Variance of Unscaled Dot Products:**
   The unscaled affinity score $S_{ij} = q_i k_j^\top = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$ has mean zero and variance proportional to the key dimension $d_k$:
   $$\mathbb{E}[S_{ij}] = 0, \quad \mathrm{Var}(S_{ij}) = d_k, \quad \sigma(S_{ij}) = \sqrt{d_k}$$

3. **Variance Normalization via $1/\sqrt{d_k}$:**
   The scaled logit $Z_{ij} = \frac{S_{ij}}{\sqrt{d_k}}$ satisfies:
   $$\mathbb{E}[Z_{ij}] = 0, \quad \mathrm{Var}(Z_{ij}) = 1$$
   independent of the projection dimension $d_k$.

4. **Softmax Degeneracy and Gradient Vanishing Without Scaling:**
   As $d_k \to \infty$, the variance of unscaled logits $S_{ij}$ diverges as $O(d_k)$. The gap between the maximal logit and other logits grows as $O(\sqrt{d_k \log m})$, driving the softmax distribution to a one-hot distribution (hard $\mathrm{argmax}$). In this saturated regime, the Jacobian $\frac{\partial \mathrm{softmax}(z)_j}{\partial z_l} = s_j (\delta_{jl} - s_l)$ approaches zero exponentially, causing vanishing gradients and stalling optimization.

---

### Step-by-Step Derivation

#### Step 1: Attention as Content-Based Memory Retrieval
In memory retrieval, a query $q_i$ seeks relevant information stored in keys $k_j$ to retrieve corresponding values $v_j$. 
The bilinear affinity between $q_i$ and $k_j$ is given by the inner product:
$$S_{ij} = \langle q_i, k_j \rangle = q_i k_j^\top = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$$
Collecting all pairs into matrix notation:
$$S = Q K^\top \in \mathbb{R}^{n \times m}$$
To form normalized attention weights summing to 1 across keys for each query row, we apply the row-wise softmax:
$$A = \mathrm{softmax}(S)$$
The retrieved values are then the weighted sum:
$$\mathrm{Attn}(Q, K, V) = A V = \mathrm{softmax}(Q K^\top) V$$

#### Step 2: Expectation and Variance of the Unscaled Dot Product
We analyze the distribution of each scalar logit $S_{ij} = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$.
Using linearity of expectation and independence between components:
$$\mathbb{E}[S_{ij}] = \mathbb{E}\left[ \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \right] = \sum_{l=1}^{d_k} \mathbb{E}[q_{i,l}] \mathbb{E}[k_{j,l}] = \sum_{l=1}^{d_k} 0 \cdot 0 = 0$$

For the variance, because the terms $q_{i,l} k_{j,l}$ are independent across dimension index $l \in \{1, \dots, d_k\}$:
$$\mathrm{Var}(S_{ij}) = \mathrm{Var}\left( \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \right) = \sum_{l=1}^{d_k} \mathrm{Var}(q_{i,l} k_{j,l})$$

For each individual term:
$$\mathrm{Var}(q_{i,l} k_{j,l}) = \mathbb{E}[(q_{i,l} k_{j,l})^2] - (\mathbb{E}[q_{i,l} k_{j,l}])^2 = \mathbb{E}[q_{i,l}^2] \mathbb{E}[k_{j,l}^2] - 0$$
Since $\mathbb{E}[q_{i,l}] = 0$ and $\mathrm{Var}(q_{i,l}) = 1$, we have $\mathbb{E}[q_{i,l}^2] = \mathrm{Var}(q_{i,l}) + (\mathbb{E}[q_{i,l}])^2 = 1 + 0 = 1$. Similarly $\mathbb{E}[k_{j,l}^2] = 1$.
Therefore:
$$\mathrm{Var}(q_{i,l} k_{j,l}) = 1 \cdot 1 = 1$$
Summing over all $d_k$ dimensions:
$$\mathrm{Var}(S_{ij}) = \sum_{l=1}^{d_k} 1 = d_k$$
The standard deviation of the unscaled inner product is therefore:
$$\sigma(S_{ij}) = \sqrt{\mathrm{Var}(S_{ij})} = \sqrt{d_k}$$

#### Step 3: Scaling by $1/\sqrt{d_k}$
To keep the scale of the logits invariant to the choice of projection dimension $d_k$, we define the scaled logit:
$$Z_{ij} = \frac{S_{ij}}{\sqrt{d_k}}$$
Its expectation is:
$$\mathbb{E}[Z_{ij}] = \mathbb{E}\left[ \frac{S_{ij}}{\sqrt{d_k}} \right] = \frac{1}{\sqrt{d_k}} \mathbb{E}[S_{ij}] = 0$$
Using the scaling property of variance $\mathrm{Var}(a X) = a^2 \mathrm{Var}(X)$:
$$\mathrm{Var}(Z_{ij}) = \mathrm{Var}\left( \frac{S_{ij}}{\sqrt{d_k}} \right) = \left(\frac{1}{\sqrt{d_k}}\right)^2 \mathrm{Var}(S_{ij}) = \frac{1}{d_k} \cdot d_k = 1$$
Furthermore, by the Lindeberg-Lévy Central Limit Theorem, as $d_k \to \infty$:
$$Z_{ij} = \frac{1}{\sqrt{d_k}} \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \xrightarrow{d} \mathcal{N}(0, 1)$$
Thus, dividing by $\sqrt{d_k}$ standardizes the dot product to standard normal distribution $\mathcal{N}(0, 1)$ regardless of how large $d_k$ is.

#### Step 4: Softmax Behavior as $d_k$ Grows Without Scaling
What happens when we omit the $1/\sqrt{d_k}$ factor and compute $\mathrm{softmax}(S_{i,:})$ directly?

1. **Logit Magnitude Growth:**
   As $d_k$ grows (e.g. $d_k = 64, 512, 1024$), the standard deviation $\sigma(S_{ij}) = \sqrt{d_k}$ increases dramatically ($\sqrt{64}=8$, $\sqrt{512}\approx 22.6$, $\sqrt{1024}=32$).
   For $m$ keys, the expected difference between the largest logit $S_{(1)}$ and the runner-up $S_{(2)}$ scales as $O(\sqrt{d_k \log m})$.

2. **Saturation to Hard Argmax:**
   Let $j^* = \operatorname{argmax}_j S_{ij}$. For any competitor $j \ne j^*$:
   $$\frac{\exp(S_{ij})}{\exp(S_{ij^*})} = \exp(S_{ij} - S_{ij^*}) \le \exp(- c \sqrt{d_k}) \to 0 \quad \text{as } d_k \to \infty$$
   Hence, the softmax probabilities saturate:
   $$s_{j^*} \to 1, \quad s_j \to 0 \quad (\forall j \ne j^*)$$
   The continuous attention mechanism collapses into an inflexible hard selection operator.

3. **Gradient Vanishing in Backpropagation:**
   Let $s = \mathrm{softmax}(z) \in \mathbb{R}^m$. The Jacobian matrix $J \in \mathbb{R}^{m \times m}$ of the softmax transformation has elements:
   $$\frac{\partial s_j}{\partial z_l} = s_j (\delta_{jl} - s_l) = \begin{cases} s_j (1 - s_j), & \text{if } j = l \\ -s_j s_l, & \text{if } j \ne l \end{cases}$$
   When the distribution is saturated:
   - For the winner $j^*$: $s_{j^*} \approx 1 \implies s_{j^*}(1 - s_{j^*}) \approx 1 \cdot 0 = 0$.
   - For all other keys $j \ne j^*$: $s_j \approx 0 \implies s_j(1 - s_j) \approx 0 \cdot 1 = 0$.
   - For cross-terms ($j \ne l$): $-s_j s_l \approx 0$.
   Therefore:
   $$\lim_{d_k \to \infty} \frac{\partial s_j}{\partial S_{il}} = 0 \quad \forall j, l$$
   By the chain rule, gradients of the loss $\mathcal{L}$ with respect to queries and keys satisfy:
   $$\frac{\partial \mathcal{L}}{\partial q_i} = \sum_{j=1}^m \frac{\partial \mathcal{L}}{\partial A_{ij}} \frac{\partial A_{ij}}{\partial q_i} \to 0$$
   The gradients vanish exponentially with respect to $\sqrt{d_k}$, halting backpropagation through the query and key projection weights. The $1/\sqrt{d_k}$ scaling maintains variance at 1, placing the logits firmly inside the sensitive, high-gradient regime of the softmax function.

---

### Sanity Checks & Edge Cases

1. **Dimensional Consistency ($d_k = 1$):**
   When $d_k = 1$, $\sqrt{d_k} = 1$. The scaling factor is $\frac{1}{\sqrt{1}} = 1$. The scalar product $q_i k_j$ already has unit variance $\mathrm{Var}(q_i k_j) = 1 \cdot 1 = 1$, and no scaling is required.
2. **Orthogonal Queries and Keys:**
   If $q_i \perp k_j$ for all $j$, $S_{ij} = 0$. Then $\exp(0 / \sqrt{d_k}) = 1$, yielding uniform attention $A_{ij} = \frac{1}{m}$ for all $j$, independent of $d_k$.
3. **Temperature Analogy:**
   The scaling $1/\sqrt{d_k}$ is mathematically equivalent to setting the Boltzmann temperature parameter $T = \sqrt{d_k}$ in statistical physics: $p_j \propto \exp(E_j / T)$. As system degrees of freedom ($d_k$) increase, temperature must scale as $\sqrt{d_k}$ to preserve thermodynamic entropy and avoid crystallization into a single ground state.


### Problem 2 [Theory] (Permutation equivariance & Positional Encodings)

**Task:**
Show that self-attention is permutation-equivariant, i.e. permuting the input rows permutes the output rows identically. Explain why this forces the use of positional encodings, and analyse the sinusoidal encoding’s ability to represent relative positions.

---

### Assumptions & Notation

1. **Input Sequence Representation:**
   Let $X \in \mathbb{R}^{n \times d}$ be an input sequence matrix where row $i \in \{1, \dots, n\}$ corresponds to token representation $x_i \in \mathbb{R}^d$ at sequence index $i$.
   The projection matrices are $W_Q, W_K \in \mathbb{R}^{d \times d_k}$, $W_V \in \mathbb{R}^{d \times d_v}$, and output projection $W_O \in \mathbb{R}^{d_v \times d_{out}}$.
   The self-attention mapping $f: \mathbb{R}^{n \times d} \to \mathbb{R}^{n \times d_{out}}$ is defined as:
   $$f(X) = \mathrm{softmax}\left(\frac{(X W_Q)(X W_K)^\top}{\sqrt{d_k}}\right) (X W_V) W_O$$

2. **Permutation Matrix:**
   Let $\pi: \{1, \dots, n\} \to \{1, \dots, n\}$ be an arbitrary permutation (bijection) of the sequence indices.
   The permutation matrix $P \in \{0, 1\}^{n \times n}$ associated with $\pi$ is defined by:
   $$P_{ij} = \begin{cases} 1, & \text{if } j = \pi(i) \\ 0, & \text{otherwise} \end{cases}$$
   Left-multiplication by $P$ permutes the rows according to $\pi$: $(P X)_i = x_{\pi(i)}$.
   $P$ is an orthogonal matrix:
   $$P^\top P = P P^\top = I_n, \quad P^{-1} = P^\top$$

3. **Sinusoidal Positional Encoding:**
   For sequence position $pos \in \{0, 1, \dots, n-1\}$ and channel index $j \in \{0, 1, \dots, d-1\}$, the sinusoidal positional encoding matrix $PE \in \mathbb{R}^{n \times d}$ is given by:
   $$PE_{(pos, 2i)} = \sin(\omega_i \cdot pos), \quad PE_{(pos, 2i+1)} = \cos(\omega_i \cdot pos)$$
   where the angular frequencies follow a geometric progression:
   $$\omega_i = \frac{1}{10000^{2i/d}}, \quad i \in \left\{0, 1, \dots, \frac{d}{2} - 1\right\}$$

---

### Formal Proposition

1. **Permutation Equivariance:**
   The self-attention operator $f$ is permutation-equivariant under any sequence permutation matrix $P$:
   $$f(P X) = P f(X)$$

2. **Necessity of Positional Encodings:**
   Because $f(P X) = P f(X)$, self-attention behaves as a function on an unordered multiset of tokens rather than an ordered sequence. Rearranging words produces the identical representations, simply permuted. Since natural language syntax and semantics depend essentially on word order, position information must be explicitly added to inputs:
   $$\tilde{X} = X + PE$$
   thereby breaking permutation equivariance so that $f(P(X + PE)) \ne P f(X + PE)$.

3. **Relative Position Representation via Linear Transformation:**
   For any integer displacement $k \in \mathbb{Z}$, there exists a fixed, position-independent orthogonal linear transformation matrix $M_k \in \mathbb{R}^{d \times d}$ such that:
   $$PE_{pos + k} = M_k PE_{pos}$$
   where $M_k$ is a block-diagonal 2D rotation matrix:
   $$M_k = \mathrm{diag}\left( R(\omega_0 k), R(\omega_1 k), \dots, R\left(\omega_{\frac{d}{2}-1} k\right) \right), \quad R(\omega_i k) \in \mathrm{SO}(2)$$
   Moreover, the inner product $\langle PE_{pos}, PE_{pos+k} \rangle = \sum_{i=0}^{d/2-1} \cos(\omega_i k)$ depends strictly on relative offset $k$.

---

### Step-by-Step Derivation

#### Step 1: Projection of Row-Permuted Inputs
Let $\tilde{X} = P X$ be the row-permuted input matrix.
Applying the linear projections:
$$\tilde{Q} = \tilde{X} W_Q = (P X) W_Q = P (X W_Q) = P Q$$
$$\tilde{K} = \tilde{X} W_K = (P X) W_K = P (X W_K) = P K$$
$$\tilde{V} = \tilde{X} W_V = (P X) W_V = P (X W_V) = P V$$
where $Q, K \in \mathbb{R}^{n \times d_k}$ and $V \in \mathbb{R}^{n \times d_v}$.

#### Step 2: Attention Logits Under Permutation
The affinity logits for the permuted input sequence are:
$$\tilde{S} = \frac{\tilde{Q} \tilde{K}^\top}{\sqrt{d_k}} = \frac{(P Q)(P K)^\top}{\sqrt{d_k}} = \frac{P Q K^\top P^\top}{\sqrt{d_k}} = P \left(\frac{Q K^\top}{\sqrt{d_k}}\right) P^\top = P S P^\top$$
At the scalar element level, entry $(i, j)$ of $\tilde{S}$ is:
$$\tilde{S}_{ij} = (P S P^\top)_{ij} = \sum_{a=1}^n \sum_{b=1}^n P_{ia} S_{ab} P_{jb} = S_{\pi(i), \pi(j)}$$
Thus, the similarity matrix $\tilde{S}$ has both its rows and columns permuted by $\pi$.

#### Step 3: Row-wise Softmax Transformation
Let $\tilde{A} = \mathrm{softmax}(\tilde{S})$.
Consider the $(i, j)$-th element of $\tilde{A}$:
$$\tilde{A}_{ij} = \frac{\exp(\tilde{S}_{ij})}{\sum_{l=1}^n \exp(\tilde{S}_{il})} = \frac{\exp(S_{\pi(i), \pi(j)})}{\sum_{l=1}^n \exp(S_{\pi(i), \pi(l)})}$$
Because $\pi: \{1, \dots, n\} \to \{1, \dots, n\}$ is a bijective mapping, summing over $l \in \{1, \dots, n\}$ visits every element of $\{1, \dots, n\}$ exactly once:
$$\sum_{l=1}^n \exp(S_{\pi(i), \pi(l)}) = \sum_{m=1}^n \exp(S_{\pi(i), m})$$
Therefore:
$$\tilde{A}_{ij} = \frac{\exp(S_{\pi(i), \pi(j)})}{\sum_{m=1}^n \exp(S_{\pi(i), m})} = A_{\pi(i), \pi(j)}$$
In matrix form:
$$\tilde{A} = \mathrm{softmax}(P S P^\top) = P \, \mathrm{softmax}(S) \, P^\top = P A P^\top$$

#### Step 4: Value Combination and Output Projection
Now, multiplying the attention matrix $\tilde{A}$ by the value matrix $\tilde{V}$:
$$\tilde{A} \tilde{V} = (P A P^\top)(P V) = P A (P^\top P) V$$
Since $P$ is an orthogonal permutation matrix, $P^\top P = I_n$:
$$\tilde{A} \tilde{V} = P A I_n V = P (A V)$$
Finally, applying the output projection $W_O$:
$$f(\tilde{X}) = (\tilde{A} \tilde{V}) W_O = (P (A V)) W_O = P (A V W_O) = P f(X)$$
This proves that self-attention is strictly permutation-equivariant: permuting input rows permutes output rows identically.

#### Step 5: Why Permutation Equivariance Forces Positional Encodings
In natural language, meaning is profoundly sensitive to word order. For example:
- *"The dog chased the cat"* vs. *"The cat chased the dog"*
- *"Not good, quite bad"* vs. *"Not bad, quite good"*

Under pure self-attention without positional information:
1. Every token interacts with every other token purely on the basis of content (vectors $x_i$).
2. The contextual representation of a word depends only on the set of other words present in the document, not where they appear.
3. Permuting the sentence merely permutes the outputs. If outputs are pooled (e.g. mean-pooled or summed for classification), the pooled vector is completely permutation-invariant:
   $$\frac{1}{n} \mathbf{1}^\top f(P X) = \frac{1}{n} (\mathbf{1}^\top P) f(X) = \frac{1}{n} \mathbf{1}^\top f(X)$$
4. Hence, self-attention without positional encodings cannot distinguish word order or syntax. Positional encodings break this symmetry by injecting position-specific coordinates into the token vectors before computing attention.

#### Step 6: Analysis of Sinusoidal Positional Encodings (Relative Shifts)
For each channel pair $(2i, 2i+1)$ at position $pos$, the 2D positional vector is:
$$p_{pos, i} = \begin{pmatrix} \sin(\omega_i \cdot pos) \\ \cos(\omega_i \cdot pos) \end{pmatrix}$$

Now consider position $pos + k$, shifted by offset $k \in \mathbb{Z}$:
$$p_{pos+k, i} = \begin{pmatrix} \sin(\omega_i(pos + k)) \\ \cos(\omega_i(pos + k)) \end{pmatrix}$$
Using angle-sum trigonometric identities:
$$\sin(\omega_i(pos + k)) = \sin(\omega_i pos)\cos(\omega_i k) + \cos(\omega_i pos)\sin(\omega_i k)$$
$$\cos(\omega_i(pos + k)) = \cos(\omega_i pos)\cos(\omega_i k) - \sin(\omega_i pos)\sin(\omega_i k)$$
Expressing this linear system in matrix form:
$$\begin{pmatrix} \sin(\omega_i(pos + k)) \\ \cos(\omega_i(pos + k)) \end{pmatrix} = \begin{pmatrix} \cos(\omega_i k) & \sin(\omega_i k) \\ -\sin(\omega_i k) & \cos(\omega_i k) \end{pmatrix} \begin{pmatrix} \sin(\omega_i pos) \\ \cos(\omega_i pos) \end{pmatrix} = R(\omega_i k) p_{pos, i}$$
where $R(\omega_i k) \in \mathrm{SO}(2)$ is a 2D Givens rotation matrix with rotation angle $-\omega_i k$.

Assembling all $d/2$ 2D subspaces into a $d \times d$ block-diagonal matrix:
$$M_k = \begin{pmatrix} R(\omega_0 k) & 0 & \dots & 0 \\ 0 & R(\omega_1 k) & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & R\left(\omega_{\frac{d}{2}-1} k\right) \end{pmatrix} \in \mathbb{R}^{d \times d}$$
Notice that:
- $M_k$ depends **only on the relative distance $k$**, and is independent of the absolute position $pos$.
- For any position $pos$, $PE_{pos + k} = M_k PE_{pos}$.
- Because each $R(\omega_i k)$ is orthogonal ($R^\top R = I$), $M_k$ is an orthogonal transformation: $M_k^\top M_k = I_d$.

#### Step 7: Inner Product and Relative Attention
When queries and keys incorporate positional encodings ($q = (x + PE) W_Q$, $k = (x' + PE') W_K$), the dot product contains positional cross-terms:
$$\langle PE_{pos}, PE_{pos + k} \rangle = \sum_{i=0}^{d/2-1} \left[ \sin(\omega_i pos)\sin(\omega_i(pos+k)) + \cos(\omega_i pos)\cos(\omega_i(pos+k)) \right]$$
Using the cosine difference identity $\cos(\alpha - \beta) = \cos \alpha \cos \beta + \sin \alpha \sin \beta$:
$$\langle PE_{pos}, PE_{pos + k} \rangle = \sum_{i=0}^{d/2-1} \cos(\omega_i(pos + k - pos)) = \sum_{i=0}^{d/2-1} \cos(\omega_i k)$$
The inner product depends solely on relative offset $k$. Because linear projections $W_Q, W_K$ can learn linear combinations of these rotated subspaces, the model can readily attend to tokens at relative offsets (e.g. "attend to 2 tokens back", "attend to previous token") independently of where in the sequence the interaction occurs.

---

### Sanity Checks & Edge Cases

1. **Identity Permutation ($P = I_n$):**
   $f(I_n X) = I_n f(X) = f(X)$, verifying trivial consistency.
2. **Zero Offset ($k = 0$):**
   When $k = 0$, $R(0) = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = I_2$, so $M_0 = I_d$.
   The inner product is $\langle PE_{pos}, PE_{pos} \rangle = \sum_{i=0}^{d/2-1} \cos(0) = d/2 = \|PE_{pos}\|_2^2$, verifying uniform vector norm across all positions.
3. **Multi-Scale Geometric Wavelengths:**
   The wavelengths $\lambda_i = \frac{2\pi}{\omega_i} = 2\pi \cdot 10000^{2i/d}$ range from $\lambda_0 = 2\pi \approx 6.28$ tokens (fine-grained, local grammatical relationships) to $\lambda_{d/2-1} = 2\pi \cdot 10000 \approx 62,831$ tokens (smooth, macroscopic document-level trends), forming a rich Fourier position basis.


### Problem 3 [Theory] (Parameter count and asymptotic compute)

**Task:**
Count the parameters and the asymptotic compute of one multi-head attention layer with $h$ heads and model dimension $d$, and compare with a dense layer of the same width. Justify the $O(n^2 d)$ cost in sequence length $n$.

---

### Assumptions & Notation

1. **Architectural Hyperparameters:**
   - Model representation dimension: $d \in \mathbb{N}_{\ge 1}$ (also denoted $d_{\mathrm{model}}$).
   - Sequence length: $n \in \mathbb{N}_{\ge 1}$ (number of tokens in the input sequence).
   - Number of attention heads: $h \in \mathbb{N}_{\ge 1}$, where $h$ divides $d$ evenly.
   - Per-head projection dimension: $d_k = d_v = \frac{d}{h}$.

2. **Multi-Head Attention (MHA) Layer Components:**
   The standard MHA layer projects input $X \in \mathbb{R}^{n \times d}$ into queries, keys, and values, performs parallel attention across $h$ heads, and projects the concatenated outputs:
   - Query projection: $W_Q \in \mathbb{R}^{d \times d}$, bias $b_Q \in \mathbb{R}^d$.
   - Key projection: $W_K \in \mathbb{R}^{d \times d}$, bias $b_K \in \mathbb{R}^d$.
   - Value projection: $W_V \in \mathbb{R}^{d \times d}$, bias $b_V \in \mathbb{R}^d$.
   - Output projection: $W_O \in \mathbb{R}^{d \times d}$, bias $b_O \in \mathbb{R}^d$.
   *(Note: Decomposing into $h$ heads corresponds to slicing each projection into $h$ blocks of shape $d \times d_k$, which is algebraically identical in parameter count and arithmetic operations).*

3. **Dense Reference Layer:**
   A standard linear/dense layer of width $d$ mapping $\mathbb{R}^d \to \mathbb{R}^d$:
   $$Y = X W + \mathbf{1} b^\top, \quad W \in \mathbb{R}^{d \times d}, \quad b \in \mathbb{R}^d$$
   Applied row-wise across the sequence length $n$.

4. **Operation Counting Metric:**
   We report both Multiply-Accumulate operations (MACs) and standard Floating Point Operations (FLOPs, where $1 \text{ MAC} \approx 2 \text{ FLOPs}$).

---

### Formal Proposition

1. **Parameter Count Comparison:**
   - **Multi-Head Attention Layer:**
     $$\text{Weights: } 4 d^2, \quad \text{Biases: } 4 d, \quad \text{Total: } 4 d^2 + 4 d$$
     *Crucially, the parameter count is independent of the number of heads $h$, provided $h \cdot d_k = d$.*
   - **Dense Layer ($d \to d$):**
     $$\text{Weights: } d^2, \quad \text{Biases: } d, \quad \text{Total: } d^2 + d$$
   - **Ratio:**
     $$\frac{\text{Params}(\mathrm{MHA})}{\text{Params}(\mathrm{Dense})} = \frac{4(d^2 + d)}{d^2 + d} = 4$$
     An MHA layer has exactly $4\times$ the parameter count of a single dense layer of the same width.

2. **Asymptotic Compute Comparison:**
   - **Multi-Head Attention Layer:**
     $$\mathcal{C}_{\mathrm{MHA}} = 4 n d^2 + 2 n^2 d \text{ MACs} \implies O(n d^2 + n^2 d)$$
   - **Dense Layer ($d \to d$):**
     $$\mathcal{C}_{\mathrm{Dense}} = n d^2 \text{ MACs} \implies O(n d^2)$$

3. **Justification of the $O(n^2 d)$ Term:**
   Every query token $i \in \{1, \dots, n\}$ computes an inner product with every key token $j \in \{1, \dots, n\}$ across all $h$ heads. The $n \times n$ attention affinity matrix requires $n^2 \cdot d_k$ MACs per head; multiplying by $h$ heads yields $h \cdot (n^2 d_k) = n^2 (h d_k) = n^2 d$. Similarly, aggregating the value vectors by the attention weights requires another $h \cdot (n^2 d_k) = n^2 d$ operations. 
   When sequence length dominates dimension ($n \gg d$), this pairwise interaction term $O(n^2 d)$ dominates the linear projection term $O(n d^2)$, causing standard self-attention to scale quadratically in compute and memory.

---

### Step-by-Step Derivation

#### Step 1: Parameter Count of Multi-Head Attention
The MHA layer consists of four affine transformations:
1. Query projection: $W_Q \in \mathbb{R}^{d \times d}$ has $d^2$ weights; $b_Q \in \mathbb{R}^d$ has $d$ biases.
2. Key projection: $W_K \in \mathbb{R}^{d \times d}$ has $d^2$ weights; $b_K \in \mathbb{R}^d$ has $d$ biases.
3. Value projection: $W_V \in \mathbb{R}^{d \times d}$ has $d^2$ weights; $b_V \in \mathbb{R}^d$ has $d$ biases.
4. Output projection: $W_O \in \mathbb{R}^{d \times d}$ has $d^2$ weights; $b_O \in \mathbb{R}^d$ has $d$ biases.

Summing all four linear blocks:
$$\text{Total Weights} = d^2 + d^2 + d^2 + d^2 = 4 d^2$$
$$\text{Total Biases} = d + d + d + d = 4 d$$
$$\text{Total Parameters} = 4 d^2 + 4 d = 4 d(d + 1)$$

*Invariance to $h$:*
If formulated as $h$ distinct head projections where $W_Q^{(i)} \in \mathbb{R}^{d \times d_k}$ with $d_k = d / h$:
$$\sum_{i=1}^h \text{Params}(W_Q^{(i)}) = h \cdot (d \cdot d_k) = h \cdot \left(d \cdot \frac{d}{h}\right) = d^2$$
Hence, splitting the representation into multiple heads redistributes the channel subspace without adding or removing a single parameter.

#### Step 2: Parameter Count of Dense Layer
A single dense layer mapping input $X \in \mathbb{R}^{n \times d}$ to output $Y \in \mathbb{R}^{n \times d}$ via $Y = X W + \mathbf{1}b^\top$:
- Weight matrix $W \in \mathbb{R}^{d \times d}$: $d^2$ parameters.
- Bias vector $b \in \mathbb{R}^d$: $d$ parameters.
$$\text{Total Parameters} = d^2 + d = d(d + 1)$$

Comparing the two:
$$\text{Params}(\mathrm{MHA}) = 4 \times \text{Params}(\mathrm{Dense})$$
*(Note: A full Transformer feedforward MLP block consists of two dense layers: $d \to 4d$ and $4d \to d$, containing $d \cdot 4d + 4d \cdot d = 8d^2$ weights, which is twice the parameter count of the MHA block).*

#### Step 3: Detailed Step-by-Step Operation Breakdown of MHA
For an input sequence of length $n$ with embedding dimension $d$:

1. **Input Projections ($Q, K, V$):**
   Computing $Q = X W_Q$, $K = X W_K$, $V = X W_V$:
   - Each projection is a matrix multiplication $(n \times d) \times (d \times d)$.
   - MACs per projection: $n \cdot d \cdot d = n d^2$.
   - For all 3 projections:
     $$\mathcal{C}_{\mathrm{proj}} = 3 n d^2 \text{ MACs} \quad (6 n d^2 \text{ FLOPs})$$

2. **Per-Head Attention Score Matrix ($Q_i K_i^\top$):**
   For head $i \in \{1, \dots, h\}$, $Q_i \in \mathbb{R}^{n \times d_k}$ and $K_i \in \mathbb{R}^{n \times d_k}$:
   - Multiplying $(n \times d_k)$ by $(d_k \times n)$ produces an $n \times n$ matrix.
   - MACs per head: $n \cdot n \cdot d_k = n^2 d_k$.
   - Across all $h$ heads:
     $$\mathcal{C}_{\mathrm{scores}} = h \cdot (n^2 d_k) = n^2 (h \cdot d_k) = n^2 d \text{ MACs} \quad (2 n^2 d \text{ FLOPs})$$

3. **Softmax Normalization:**
   Applying $\mathrm{softmax}\left(\frac{S_i}{\sqrt{d_k}}\right)$ row-wise over each $n \times n$ matrix:
   - For each head: scaling, exponentiation, sum reduction, and division over $n \times n$ entries costs $O(n^2)$ FLOPs.
   - Across all $h$ heads: $O(h n^2)$ FLOPs.

4. **Context Aggregation ($A_i V_i$):**
   For head $i$, multiplying attention matrix $A_i \in \mathbb{R}^{n \times n}$ by values $V_i \in \mathbb{R}^{n \times d_k}$:
   - Matrix product $(n \times n) \times (n \times d_k) \implies n \cdot n \cdot d_k = n^2 d_k$ MACs.
   - Across all $h$ heads:
     $$\mathcal{C}_{\mathrm{values}} = h \cdot (n^2 d_k) = n^2 (h \cdot d_k) = n^2 d \text{ MACs} \quad (2 n^2 d \text{ FLOPs})$$

5. **Output Linear Projection ($W_O$):**
   Concatenating head outputs yields $H \in \mathbb{R}^{n \times d}$. Multiplying $H W_O$:
   - Matrix product $(n \times d) \times (d \times d) \implies n \cdot d \cdot d = n d^2$ MACs.
     $$\mathcal{C}_{\mathrm{out}} = n d^2 \text{ MACs} \quad (2 n d^2 \text{ FLOPs})$$

#### Step 4: Total Asymptotic Complexity Summation
Summing all terms:
$$\mathcal{C}_{\mathrm{MHA}} = \underbrace{3 n d^2}_{\text{Q, K, V proj}} + \underbrace{n^2 d}_{\text{scores } QK^\top} + \underbrace{O(h n^2)}_{\text{softmax}} + \underbrace{n^2 d}_{\text{aggregation } AV} + \underbrace{n d^2}_{\text{output proj}}$$
$$\mathcal{C}_{\mathrm{MHA}} = 4 n d^2 + 2 n^2 d + O(h n^2) \text{ MACs}$$
In standard Big-$O$ notation:
$$\mathcal{C}_{\mathrm{MHA}} = O(n d^2 + n^2 d)$$

#### Step 5: Dense Layer Complexity
For a dense layer $Y = X W + b$ on input $X \in \mathbb{R}^{n \times d}$:
- Matrix product $(n \times d) \times (d \times d) \implies n d^2$ MACs.
$$\mathcal{C}_{\mathrm{Dense}} = n d^2 \text{ MACs} \implies O(n d^2)$$

#### Step 6: Regime Analysis & Justification of the $O(n^2 d)$ Bottleneck
We compare how compute scales across different operating regimes:

| Regime | Condition | Dominant MHA Term | Asymptotic Scaling | MHA vs. Dense Compute Ratio |
|---|---|---|---|---|
| **Short context** | $n \ll d$ | $4 n d^2$ | $O(n d^2)$ | $\approx 4\times$ |
| **Balanced** | $n \approx 2d$ | $4 n d^2 \approx 2 n^2 d$ | $O(n d^2) = O(n^2 d)$ | $\approx 6\times$ |
| **Long context** | $n \gg d$ | $2 n^2 d$ | $O(n^2 d)$ | $\approx \frac{2n}{d} \gg 1$ |

**Why the $O(n^2 d)$ cost arises:**
1. A dense layer treats each token independently. There is zero token-to-token communication; the computational graph has $n$ parallel, disconnected paths, resulting in linear cost $O(n d^2)$.
2. Multi-head attention explicitly models all-to-all pairwise interactions between tokens. To allow any token to attend to any other token, the mechanism must construct an $n \times n$ affinity graph.
3. Every cell $(i, j)$ in that affinity matrix represents an inner product of dimension $d_k$ across $h$ heads, guaranteeing that exactly $h \cdot d_k = d$ operations are performed per pair.
4. Hence, the $O(n^2 d)$ cost is the fundamental price of dense, full-context attention. For long sequences (e.g. $n = 32,768$ or $128,000$), $n^2$ grows prohibitively large, motivating linear-attention approximations, FlashAttention (to avoid materializing the $O(n^2)$ matrix in HBM), and sliding-window attention mechanisms.

---

### Sanity Checks & Edge Cases

1. **Extreme Head Counts:**
   - **Single Head ($h = 1, d_k = d$):** $\text{Params} = 4d^2 + 4d$; MACs $= 4nd^2 + 2n^2 d$.
   - **Maximal Heads ($h = d, d_k = 1$):** Each head operates on scalars ($d_k=1$). $\text{Params} = 4d^2 + 4d$; MACs $= 4nd^2 + 2n^2 d$. Parameter and arithmetic costs remain identical.
2. **Autoregressive Decoding ($n = 1$ with KV Cache):**
   When generating a single token given a precomputed Key-Value cache of length $T$:
   - Projecting single query $q \in \mathbb{R}^{1 \times d}$: $d^2$ MACs.
   - Attention score with cached keys $q K^\top$: $(1 \times d) \times (d \times T) \implies T d$ MACs.
   - Aggregation with cached values $A V$: $(1 \times T) \times (T \times d) \implies T d$ MACs.
   - Output projection: $d^2$ MACs.
   - Compute is $O(d^2 + T d)$, strictly linear in cache length $T$.


### Problem 4 [Python]

**Task:**
Implement scaled dot-product attention and multi-head attention from scratch and check your output against torch.nn.MultiheadAttention on random inputs (max abs. error < 10^-5).


In [ ]:
# Problem 4 [Python]
import torch
import torch.nn as nn

# TODO: Implement scaled dot-product & MultiheadAttention from scratch and verify against torch.nn.MultiheadAttention


### Problem 5 [Python]

**Task:**
Use a pretrained HuggingFace transformer two ways: (a) as a frozen sentence-embedding extractor feeding a linear classifier, and (b) fine-tuned end-to-end. Compare both against the TF-IDF baseline from Problem Set 3 on the same data.


In [ ]:
# Problem 5 [Python]
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch

# TODO: (a) Frozen sentence-embedding extractor + linear classifier
#       (b) Fine-tuned end-to-end
#       Compare with PS3 TF-IDF


### Problem 6 [Analysis]

---
**Visualisation & Interpretation:**
